# EODH Daily Origin-Destination Matrices

This operational notebook builds sample-count matrices from the validated EM2023 relational database. It publishes a complete daily OD matrix and a separate matrix by the source main-mode category. Expansion remains gated on field-level documentary validation of `fexp_vj`. EODI is a separate workstream and is not used to calibrate these products.


## 1. Product contract

- **Observation unit:** one record in the SQLite `viajes` table.
- **Geography:** the 2023 ZAT attribute table. A zone is in scope when its municipality code is Bogota or one of the 20 municipalities in the official EODH study area. Other mapped gateway or municipality zones are retained in reconciliation, not folded into the in-scope matrix.
- **Main matrix:** daily sample trip counts by origin and destination ZAT, including intrazonal trips and trips with an unclassified mode.
- **Mode matrix:** same universe, grouped by the exact source value in `modo_principal_agrupado`; blank modes would be reported as `[unclassified]`.
- **Expansion:** not applied. Official methodology describes a doubly robust model-based estimator, but reviewed documentation does not establish the field-level derivation and operational universe of `fexp_vj`. No uncertainty estimates are produced because the model-based variance procedure for OD-by-mode cells is not reproducible from the reviewed materials.

The outputs contain aggregated zone counts only. Trip, person, and household identifiers are used only for validation and are never written to the products.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").is_file() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Could not locate the project root.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from mode_choice.od_matrix import build_od_products


## 2. Validate and build products

The builder opens SQLite read-only, validates the trip key and ZAT codes, checks that the mode totals reconcile, and stages all reports before replacing the prior output directory. If any input or validation fails, previously published products remain available.


In [2]:
products = build_od_products(PROJECT_ROOT)
output_dir = products["output_directory"]
print(f"Published sample OD products: {output_dir.relative_to(PROJECT_ROOT)}")
print(f"Weighted matrices published: {products['weighted_products_published']}")
display(products["geography"])
display(products["summary"])
display(products["factor"])

Published sample OD products: data\analytical\od_matrices
Weighted matrices published: False


{'zone_count': 1215,
 'study_area_zone_count': 1199,
 'external_or_gateway_zone_count': 16}

,metric,count
0,source_trip_rows,100174
1,unique_trip_keys,100174
2,sample_matrix_in_scope_trips,99741
3,sample_trips_with_classified_source_mode,99741
4,sample_trips_with_unclassified_mode,0
5,intrazonal_sample_trips,15013
6,sample_od_pairs,47192
7,sample_od_mode_pairs,59957
8,destination_external_or_gateway_zone,196
9,destination_study_area_zone,99978


,source_trip_rows,factor_nonempty_rows,factor_numeric_rows,factor_non_numeric_rows,factor_missing_or_blank_rows,factor_negative_rows,factor_zero_rows,factor_sum_diagnostic_not_applied,expansion_release_status,expansion_documentation_validated
0,100174,100174,100174,0,0,0,0,16723043.3,not_published_documentary_validation_pending,False


## 3. Read the sparse matrices

Each row represents one observed origin-destination pair. The mode table adds the exact source mode category. No dense zero-filled matrix is generated; downstream tools can pivot these sparse tables when needed.


In [3]:
import pandas as pd

total_matrix = pd.read_csv(output_dir / "eodh_sample_od_total.csv")
mode_matrix = pd.read_csv(output_dir / "eodh_sample_od_by_mode.csv")
reconciliation = pd.read_csv(output_dir / "eodh_sample_reconciliation.csv")
geo_status = pd.read_csv(output_dir / "eodh_sample_geography_pair_status.csv")
manifest = pd.read_json(output_dir / "manifest.json", typ="series")

assert int(total_matrix["sample_trips"].sum()) == int(reconciliation.loc[reconciliation.metric == "sample_matrix_in_scope_trips", "count"].iloc[0])
assert int(mode_matrix["sample_trips"].sum()) == int(total_matrix["sample_trips"].sum())
mode_by_od = mode_matrix.groupby(["zat_ori", "zat_des"], as_index=False)["sample_trips"].sum()
reconciled_by_od = total_matrix.merge(mode_by_od, on=["zat_ori", "zat_des"], suffixes=("_total", "_by_mode"), validate="one_to_one")
assert (reconciled_by_od["sample_trips_total"] == reconciled_by_od["sample_trips_by_mode"]).all()
assert not total_matrix.duplicated(["zat_ori", "zat_des"]).any()
assert not mode_matrix.duplicated(["zat_ori", "zat_des", "modo_principal_agrupado"]).any()
assert total_matrix["zat_ori"].notna().all() and total_matrix["zat_des"].notna().all()
assert "key_viaje" not in total_matrix.columns and "key_viaje" not in mode_matrix.columns

display(total_matrix.head(10))
display(mode_matrix.head(10))
display(reconciliation)
display(geo_status)
display(manifest.to_frame("value"))

,zat_ori,zat_des,sample_trips
0,1,1070,1
1,1,134,2
2,1,150,1
3,1,48,1
4,10,10,14
5,10,1001,1
6,10,1011,1
7,10,1012,2
8,10,1022,3
9,10,1059,1


,zat_ori,zat_des,modo_principal_agrupado,sample_trips
0,1,1070,BICICLETA,1
1,1,134,TAXI OCUPADO,2
2,1,150,TRANSPORTE ESCOLAR,1
3,1,48,AUTO,1
4,10,10,A PIE <15 MIN,7
5,10,10,A PIE > 15 MIN,7
6,10,1001,TRANSPORTE PÚBLICO,1
7,10,1011,AUTO,1
8,10,1012,BICICLETA,2
9,10,1022,AUTO,3


,metric,count
0,source_trip_rows,100174
1,unique_trip_keys,100174
2,sample_matrix_in_scope_trips,99741
3,sample_trips_with_classified_source_mode,99741
4,sample_trips_with_unclassified_mode,0
5,intrazonal_sample_trips,15013
6,sample_od_pairs,47192
7,sample_od_mode_pairs,59957
8,destination_external_or_gateway_zone,196
9,destination_study_area_zone,99978


,pair_status,trip_rows
0,external_or_gateway_zone__to__external_or_gate...,3
1,external_or_gateway_zone__to__study_area_zone,237
2,study_area_zone__to__external_or_gateway_zone,193
3,study_area_zone__to__study_area_zone,99741


,value
product,EM2023 EODH sample daily OD matrices
status,sample_products_published_expansion_pending_do...
source_database,data/relational/em2023_relational.sqlite
source_table,viajes
source_row_count,100174
trip_key_unique,True
zone_source,data/sources/raw/zonification/ZAT2023.dbf
zone_count,1215
official_release_page,https://observatorio.movilidadbogota.gov.co/pu...
official_release_review_date,2026-11-02


## 4. Interpretation and remaining limits

The in-scope matrix counts records, not population trips. Its total therefore must not be presented as a population estimate. The source factor is retained only in the factor diagnostic; its sum is explicitly diagnostic and is not an expanded estimate. The geography rule includes ZATs assigned to the named study municipalities, not every municipality listed in trip endpoint text. External and gateway destinations remain visible in the pair-status reconciliation.

Next, review the eligibility and geographic reconciliation and confirm this sample product as the working OD base. A weighted release requires an official field definition for `fexp_vj`, its record-level application and universe, and a compatible validation benchmark. EODI / Stage V integration remains a separate, later product.
